In [1]:
import numpy as np
from openxai.dataloader import ReturnLoaders
from openxai.model import LoadModel
from goodpoints import compress
import pandas as pd
import shap

In [2]:
_, loader_test = ReturnLoaders(data_name="gaussian", download=False, batch_size=128)
X_test = loader_test.dataset.data

model = LoadModel(data_name="gaussian", ml_model="ann", pretrained=True)
model.eval()

ArtificialNeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=20, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=100, bias=True)
    (3): ReLU()
    (4): Linear(in_features=100, out_features=2, bias=True)
  )
)

In [3]:
n = X_test.shape[0]
d = X_test.shape[1]
sigma = np.sqrt(2 * d)

gaussian and multi_quadric

In [4]:
id_cte_gaussian = compress.compresspp_kt(X_test, kernel_type=b"gaussian", k_params=np.array([sigma**2]), g=4, seed=0)
X_cte_gaussian = X_test[id_cte_gaussian]

In [5]:
id_cte_multi_quadric = compress.compresspp_kt(X_test, kernel_type=b"multi_quadric", k_params=np.array([sigma**2]), g=4, seed=0)
X_multi_quadric = X_test[id_cte_multi_quadric]

In [6]:
masker_cte_gaussian = shap.maskers.Independent(X_cte_gaussian, max_samples=X_cte_gaussian.shape[0])
explainer_cte_gaussian = shap.PermutationExplainer(lambda x: model.predict_proba(x)[:, 1], masker_cte_gaussian, seed=0)
shap_cte_gaussian = explainer_cte_gaussian(X_test)

In [7]:
masker_cte_multi_quadric = shap.maskers.Independent(X_multi_quadric, max_samples=X_multi_quadric.shape[0])
explainer_cte_multi_quadric = shap.PermutationExplainer(lambda x: model.predict_proba(x)[:, 1], masker_cte_multi_quadric, seed=0)
shap_cte_multi_quadric = explainer_cte_multi_quadric(X_test)

iid sampling

In [8]:
np.random.seed(0)
id_iid = np.random.choice(n, size=len(id_cte_gaussian))
X_iid = X_test[id_iid]

In [9]:
masker_iid = shap.maskers.Independent(X_iid, max_samples=X_iid.shape[0])
explainer_iid = shap.PermutationExplainer(lambda x: model.predict_proba(x)[:, 1], masker_iid, seed=0)
shap_iid = explainer_iid(X_test)

ground truth

In [10]:
masker_gt = shap.maskers.Independent(X_test, max_samples=X_test.shape[0])
explainer_gt = shap.PermutationExplainer(lambda x: model.predict_proba(x)[:, 1], masker_gt, seed=0)
shap_gt = explainer_gt(X_test)

PermutationExplainer explainer: 1251it [04:48,  4.17it/s]                          


compare with ground truth

In [11]:
def metric_mae(x, y):
    return np.mean(np.abs(x-y))

In [ ]:
print(f'Explanation approximation error introduced by CTE using Gaussian kernel:\
        {metric_mae(shap_gt.values, shap_cte_gaussian.values):.4f}')
print(f'Explanation approximation error introduced by CTE using inverse multiquadric kernel:\
        {metric_mae(shap_gt.values, shap_cte_multi_quadric.values):.4f}')
print(f'Explanation approximation error introduced by iid sampling:\
        {metric_mae(shap_gt.values, shap_iid.values):.4f}')


Explanation approximation error introduced by CTE using Gaussian kernel:        0.0086
Explanation approximation error introduced by CTE using multi quadric kernel:        0.0083
Explanation approximation error introduced by iid sampling:        0.0121
